In [1]:
!git clone https://github.com/NYUSH-ML/ml-competition-sp26.git

fatal: destination path 'ml-competition-sp26' already exists and is not an empty directory.


In [2]:
%cd ml-competition-sp26

/content/ml-competition-sp26


In [3]:
!pip install -r requirements.txt

In [5]:
!python download_data.py --start 20250101 --end 20260421

>> Fetching CSI500 constituents...
   500 constituents from CSI500 index.
>> Full download from 20250101 to 20260421
>> Fetching CSI500 index benchmark...
   313 index rows.
>> Fetching constituent OHLCV...
100% 499/500 [17:46<00:02,  2.39s/it]  [warn] sh689009 failed after 3 tries: No value to decode
100% 500/500 [17:56<00:00,  2.15s/it]
>> Saved 155,784 rows across 499 stocks to data/prices.parquet
   success: 499, failed: 1
>> Saved 499 constituents to data/constituents.csv (filtered to downloaded)
   dropped from universe (no price data): ['689009']


In [77]:
!pip install lightgbm catboost

  Using cached catboost-1.2.10-cp312-cp312-manylinux2014_x86_64.whl.metadata (1.2 kB)
Using cached catboost-1.2.10-cp312-cp312-manylinux2014_x86_64.whl (97.1 MB)


### Data Set Refreshing

In [28]:
# 1) Refresh data up to May 2, 2026
%cd ml-competition-sp26
!python download_data.py --update --end 20260502

[Errno 2] No such file or directory: 'ml-competition-sp26'
/content/ml-competition-sp26/ml-competition-sp26
>> Fetching CSI500 constituents...
   500 constituents from CSI500 index.
>> Incremental update from 20260422 to 20260502
>> Fetching CSI500 index benchmark...
   320 index rows.
>> Fetching constituent OHLCV...
100% 499/500 [18:56<00:02,  2.48s/it]  [warn] sh689009 failed after 3 tries: No value to decode
100% 500/500 [19:06<00:00,  2.29s/it]
>> Saved 159,277 rows across 499 stocks to data/prices.parquet
   success: 499, failed: 1
>> Saved 499 constituents to data/constituents.csv (filtered to downloaded)
   dropped from universe (no price data): ['689009']


In [57]:
# 2) Reload updated data
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
prices = pd.read_parquet(DATA_DIR / "prices.parquet")
index_df = pd.read_parquet(DATA_DIR / "index.parquet")

print("latest price date:", prices["date"].max())
print("latest index date:", index_df["date"].max())

latest price date: 2026-04-30 00:00:00
latest index date: 2026-04-30 00:00:00


### 1. Configuration & Data Pipeline

In [215]:
# ==========================================
# 1. CONFIGURATION & DATA PIPELINE
# ==========================================
import numpy as np
import pandas as pd
import xgboost as xgb

print("Initializing Configuration and Data Pipeline...")

CONFIG = {
    'random_seed': 42,
    'xgb_params': {
        'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 4,
        'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'objective': 'rank:pairwise', 'eval_metric': 'ndcg', 'tree_method': 'hist',
        'early_stopping_rounds': 50
    },
    'oos_test_days': 20,
    'val_days': 15,
    'embargo_days': FORWARD_HORIZON,

    # --- Portfolio Allocation ---
    'decay_factor': 0.001
    # 0.15 is too high, 0.03 is high, 0.02 is safe, 0.01 is great, 0.005 is even better, 0.001 better than previous, 0.0005 no significant progress.
    # Increase for more aggressive top-heavy weighting, decrease for potentially better performance in this case.
}

# --- 1B. Strict Chronological Split ---
FULL_PANEL = panel.copy()
labeled_dates = np.sort(panel.dropna(subset=[TARGET_COLUMN])['date'].unique())

test_start = labeled_dates[-CONFIG['oos_test_days']]
val_end = labeled_dates[-(CONFIG['oos_test_days'] + CONFIG['embargo_days'] + 1)]
val_start = labeled_dates[-(CONFIG['oos_test_days'] + CONFIG['embargo_days'] + CONFIG['val_days'])]
train_end = labeled_dates[-(CONFIG['oos_test_days'] + 2 * CONFIG['embargo_days'] + CONFIG['val_days'] + 1)]

pools = {
    'train': panel[panel['date'] <= train_end].copy().sort_values('date'),
    'val': panel[(panel['date'] >= val_start) & (panel['date'] <= val_end)].copy().sort_values('date'),
    'test': panel[panel['date'] >= test_start].copy().sort_values('date')
}

# --- 1C. Cross-Sectional Ranking & Target Prep ---
FEAT_COLS = feature_sets['baseline']
for name, pool in pools.items():
    # 1. Rank features to make them market-neutral
    pool[FEAT_COLS] = pool.groupby('date')[FEAT_COLS].rank(pct=True)

    # 2. Convert continuous target to 5-level integer bins for XGBRanker
    pool['target_int'] = pool.groupby('date')[TARGET_COLUMN].transform(
        lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')
    ).fillna(0).astype(int)

print(f"Data Pipeline Complete! OOS Test window starts on {pd.Timestamp(test_start).strftime('%Y-%m-%d')}")

Initializing Configuration and Data Pipeline...
Data Pipeline Complete! OOS Test window starts on 2026-03-17


### 2. Feature Sub-Ensemble Training

In [216]:
# ==========================================
# 2. FEATURE SUB-ENSEMBLE TRAINING
# ==========================================
print("Training Domain-Specific Feature Experts...")

# Query IDs for pairwise ranking
tr_qid = pools['train'].groupby('date').ngroup().values
va_qid = pools['val'].groupby('date').ngroup().values

# Dynamic Feature Grouping
feature_subsets = {
    'Momentum_Expert': [c for c in FEAT_COLS if any(x in c.lower() for x in ['ret', 'mom', 'roc', 'price', 'close'])],
    'Volume_Expert': [c for c in FEAT_COLS if any(x in c.lower() for x in ['vol', 'turnover', 'amount', 'liquidity'])],
}
# Catch-all for remaining features
feature_subsets['Tech_Expert'] = [c for c in FEAT_COLS if c not in feature_subsets['Momentum_Expert'] and c not in feature_subsets['Volume_Expert']]

trained_experts = {}

for expert_name, subset_cols in feature_subsets.items():
    if not subset_cols: continue
    print(f" -> Training {expert_name} ({len(subset_cols)} features)...")

    model = xgb.XGBRanker(**CONFIG['xgb_params'], random_state=CONFIG['random_seed'])
    model.fit(
        pools['train'][subset_cols], pools['train']['target_int'], qid=tr_qid,
        eval_set=[(pools['val'][subset_cols], pools['val']['target_int'])], eval_qid=[va_qid],
        verbose=200 # Print less often to keep output clean
    )
    trained_experts[expert_name] = {'model': model, 'cols': subset_cols}

print("\nSub-Ensemble Training Complete!")

Training Domain-Specific Feature Experts...
 -> Training Momentum_Expert (9 features)...
[0]	validation_0-ndcg:0.35657
[55]	validation_0-ndcg:0.37047
 -> Training Volume_Expert (4 features)...
[0]	validation_0-ndcg:0.36468
[51]	validation_0-ndcg:0.34441
 -> Training Tech_Expert (1 features)...
[0]	validation_0-ndcg:0.30043
[77]	validation_0-ndcg:0.36066

Sub-Ensemble Training Complete!


### 3. Helper Function & OOS Evaluation

In [217]:
# ==========================================
# 3. HELPER FUNCTION & OOS EVALUATION
# ==========================================
from score_submission import score_window

# --- 3A. Scoring Helper Function ---
def generate_ensemble_scores(df_market, experts_dict):
    """Generates cross-sectionally ranked ensemble predictions for a given market dataframe."""
    df = df_market.copy()
    score_cols = []
    for expert, params in experts_dict.items():
        col_name = f'score_{expert}'
        df[col_name] = params['model'].predict(df[params['cols']])
        df[col_name] = df.groupby('date')[col_name].rank(pct=True) # Rank to standardize
        score_cols.append(col_name)

    df['ensemble_score'] = df[score_cols].mean(axis=1)
    return df

# --- 3B. Custom Conviction Allocator ---
def make_conviction_portfolio(scores, top_k=50, max_w=0.10, decay_factor=CONFIG['decay_factor']):
    """
    Allocates capital based on exponential decay of the model's rank.
    Heavily favors the Top 10 stocks while satisfying the minimum 30 stock rule.
    """
    # 1. Sort stocks by score descending
    sorted_scores = scores.sort_values(ascending=False).head(top_k)

    # 2. Create exponential weights based on RANK, not raw score
    # e.g., e^(-0.1 * 1), e^(-0.1 * 2)...
    ranks = np.arange(1, len(sorted_scores) + 1)
    raw_weights = np.exp(-decay_factor * ranks)

    # 3. Normalize to sum to 1.0
    weights = raw_weights / raw_weights.sum()

    # 4. Enforce max weight constraint (10%)
    # If any weight > 10%, clip it and redistribute the excess proportionally
    while np.any(weights > max_w + 1e-6):
        excess = np.sum(weights[weights > max_w] - max_w)
        weights[weights > max_w] = max_w

        # Redistribute excess to stocks below the cap
        uncapped_mask = weights < max_w
        if np.any(uncapped_mask):
            weights[uncapped_mask] += excess * (weights[uncapped_mask] / weights[uncapped_mask].sum())

    # Return as pandas Series
    return pd.Series(weights, index=sorted_scores.index, name='weight')

# --- 3C. Out-Of-Sample Evaluation ---
print("Running Out-Of-Sample (OOS) Baseline Comparison...")

# Extract strict 5-day OOS window
trading_dates = np.sort(panel['date'].unique())
test_start_idx = np.where(trading_dates == test_start)[0][0]
TEST_END = pd.Timestamp(trading_dates[test_start_idx + FORWARD_HORIZON - 1])
AS_OF = pd.Timestamp(trading_dates[test_start_idx - 1])

# Ensure test_start is a Timestamp for the scoring function
TEST_START_TS = pd.Timestamp(test_start)

# 1. Score Official Baseline
!python baseline_xgboost.py --as-of {AS_OF.strftime('%Y%m%d')} --top-k 50 --out repo_baseline_sub.csv
repo_df = pd.read_csv("repo_baseline_sub.csv", dtype={"stock_code": str})
repo_metrics = score_window(repo_df.set_index("stock_code")["weight"].astype(float), prices, index_df, TEST_START_TS, TEST_END)

# 2. Score Our Ensemble
as_of_market = panel[panel['date'] == AS_OF].dropna(subset=FEAT_COLS).copy()
as_of_market[FEAT_COLS] = as_of_market[FEAT_COLS].rank(pct=True) # Apply cross-sectional ranking first

scored_market = generate_ensemble_scores(as_of_market, trained_experts)
ens_weights = make_conviction_portfolio(scored_market.set_index('stock_code')['ensemble_score'], top_k=50, max_w=0.10, decay_factor=CONFIG['decay_factor'])
ens_metrics = score_window(ens_weights, prices, index_df, TEST_START_TS, TEST_END)

# 3. Display Results
comp_df = pd.DataFrame([
    {"Model": "Feature Sub-Ensemble", "Portfolio Return": ens_metrics['portfolio_return'], "Excess Return": ens_metrics['excess_return']},
    {"Model": "Official Repo Baseline", "Portfolio Return": repo_metrics['portfolio_return'], "Excess Return": repo_metrics['excess_return']}
])
display(comp_df)

diff_bps = (comp_df.iloc[0]['Excess Return'] - comp_df.iloc[1]['Excess Return']) * 10000
print(f"Sub-Ensemble vs Repo Baseline: {diff_bps:+.2f} BPS")

Running Out-Of-Sample (OOS) Baseline Comparison...
>> Loading /content/ml-competition-sp26/ml-competition-sp26/data/prices.parquet
   159,277 rows, 499 stocks, dates 2025-01-02 to 2026-04-30
>> Building features
   train: 103,422 rows up to 2026-02-06
   embargo: 5 trading days (discarded)
   val:   4,975 rows from 2026-02-24
>> Training XGBoost
   validation rank IC: -0.0969
>> Predicting portfolio
   as of 2026-03-16, scoring 499 stocks
>> Wrote 50 names to repo_baseline_sub.csv
   weight summary: min=0.0008 max=0.0392 sum=1.0000


,Model,Portfolio Return,Excess Return
0,Feature Sub-Ensemble,-0.093965,-0.003019
1,Official Repo Baseline,-0.111314,-0.020368


Sub-Ensemble vs Repo Baseline: +173.48 BPS


### 4. Final Submission Generation


In [218]:
# ==========================================
# 4. FINAL SUBMISSION GENERATION
# ==========================================
print("Generating Final Submission Portfolio for Gradescope...")

# 1. Get the very latest day of data
latest_date = FULL_PANEL['date'].max()
current_market = FULL_PANEL[FULL_PANEL['date'] == latest_date].dropna(subset=FEAT_COLS).copy()

if current_market.empty:
    print(f"Error: No features found for {latest_date.strftime('%Y-%m-%d')}!")
else:
    # 2. Apply cross-sectional ranking to features (Crucial!)
    current_market[FEAT_COLS] = current_market[FEAT_COLS].rank(pct=True)

    # 3. Generate scores and build portfolio using our helper function
    scored_latest = generate_ensemble_scores(current_market, trained_experts)

    # 4. Create Portfolio (Using the Rank-based Top 50 method)
    weights = make_conviction_portfolio(scored_latest.set_index('stock_code')['ensemble_score'], top_k=50, max_w=0.10, decay_factor=CONFIG['decay_factor'])

    # 5. Build and Save Submission DataFrame
    submission = weights.reset_index()
    submission.columns = ['stock_code', 'weight']
    submission['stock_code'] = submission['stock_code'].astype(str).str.zfill(6)
    submission = submission.sort_values('weight', ascending=False).reset_index(drop=True)

    submission.to_csv('submission_final_for_real.csv', index=False)
    print(f"\nSaved submission_final_for_real.csv for {latest_date.strftime('%Y-%m-%d')}!")
    display(submission.head(5))

    # 6. Safety Check - Run the official competition validator!
    print("\nRunning Official Validation Check...")
    !python validate_submission.py submission_final_for_real.csv

Generating Final Submission Portfolio for Gradescope...

Saved submission_final_for_real.csv for 2026-04-21!


,stock_code,weight
0,603885,0.020494
1,600060,0.020473
2,600499,0.020453
3,600312,0.020433
4,688281,0.020412



Running Official Validation Check...
OK: submission_final_for_real.csv passes all checks
